# IDS-BGWO-SHAP — Reduced-budget run (~3 hours on a Colab T4)

Multi-seed run with a smaller BGWO budget than `run_colab.ipynb`'s paper-grade defaults.
Statistically valid for Wilcoxon signed-rank (n_seeds=5 still has 31 permutations).

**Budget vs paper-grade:**

| Knob | Paper-grade (`run_colab.ipynb`) | This notebook |
|---|---|---|
| `n_seeds` | 10 | **5** |
| `bgwo_population` | 15 | **10** |
| `bgwo_iterations` | 30 | **20** |
| `sample_size` | 500,000 | 500,000 (unchanged) |
| ETA | 10–16 h | **~3 h** |

Everything else (alpha/beta/gamma, SHAP background, SMOTE caps, fixed LCCDE downstream) is
left at paper-grade defaults — only the search budget is scaled down.

Before running this, confirm `notebooks/smoke_test.ipynb` finished cleanly. That catches
any pipeline regression in 60 seconds, not 3 hours.

## 1. Clone the repo and install dependencies

In [ ]:
import pathlib, subprocess, sys

REPO_DIR = 'IDS-features-selection'
if not pathlib.Path(REPO_DIR).exists():
    !git clone -q https://github.com/yazanjer/IDS-features-selection.git
%cd $REPO_DIR

print('Installing requirements (suppressing pip resolver chatter) ...')
proc = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check',
     '-r', 'requirements.txt'],
    capture_output=True, text=True,
)
if proc.returncode != 0:
    print('PIP INSTALL FAILED — full output below:')
    print(proc.stdout)
    print(proc.stderr)
    raise SystemExit(1)
print('Install OK.')

import importlib
for pkg in ('numpy', 'pandas', 'sklearn', 'scipy', 'lightgbm', 'xgboost',
            'catboost', 'shap', 'lime', 'imblearn', 'kagglehub'):
    m = importlib.import_module(pkg)
    print(f'  {pkg:12s} {getattr(m, "__version__", "?")}')
print('Environment ready.')

## 2. Upload `kaggle.json`

Needed to fetch the full CIC-IDS2017 dataset (~2.5 M flows) from Kaggle. Get the token
from https://www.kaggle.com/settings → API → Create New Token.

In [ ]:
from google.colab import files
from src.data_loader import install_kaggle_credentials_from_upload

print('Select kaggle.json from your local machine:')
uploaded = files.upload()
install_kaggle_credentials_from_upload(uploaded)

## 3. Configure the reduced-budget run

Statistical validity rationale for `n_seeds=5`: Wilcoxon signed-rank with n=5 has 2^5=32
sign permutations under H0, so the smallest achievable two-sided p-value is
≈ 2/32 = 0.0625. That's enough to flag practically significant effects (Cohen's d > 0.8)
but borderline for the 0.05 threshold; if the headline delta is marginal, re-run with
`n_seeds=10` (paper-grade) before publishing.

In [ ]:
from src.config import Config

cfg = Config(
    dataset='cicids2017',
    n_seeds=5,
    bgwo_population=10,
    bgwo_iterations=20,
    # Everything else stays at paper-grade defaults:
    #   sample_size=500_000, alpha=0.85, beta=0.10, gamma=0.05,
    #   fs_train_rows=30_000, fs_test_rows=10_000,
    #   shap_background_samples=100, smote_min_count=500,
    #   compute_shap_in_matrix=False (SHAP signatures computed after the matrix).
)
cfg.ensure_dirs()
print(cfg.to_json())

## 4. Run the matrix — 8 methods × 5 seeds = 40 trials

Per-trial results stream to `results/runs/` as they finish, so a Colab disconnect mid-run
doesn't lose completed work. To resume after a disconnect, re-run the install and config
cells; the in-process dataset cache will be cold but the trial JSONs persist.

In [ ]:
from src.evaluation import run_experiment_matrix, aggregate, wilcoxon_vs_reference, DEFAULT_METHODS

# DEFAULT_METHODS = ('none', 'filter', 'rfe', 'lasso', 'rf_imp', 'boruta', 'bgwo_bi', 'bgwo_shap')
df = run_experiment_matrix(cfg, methods=DEFAULT_METHODS)
df.to_csv(cfg.results_dir / 'reduced_runs.csv', index=False)

agg = aggregate(df)
agg.to_csv(cfg.results_dir / 'reduced_aggregate.csv', index=False)
agg

## 5. Wilcoxon signed-rank: `bgwo_shap` vs every other method

In [ ]:
import pandas as pd

wil_f1   = wilcoxon_vs_reference(df, 'bgwo_shap', 'macro_f1')
wil_acc  = wilcoxon_vs_reference(df, 'bgwo_shap', 'accuracy')
wil_size = wilcoxon_vs_reference(df, 'bgwo_shap', 'n_features_selected')
wil_lat  = wilcoxon_vs_reference(df, 'bgwo_shap', 'latency_ms_per_flow')

wilcoxon_df = pd.concat([wil_f1, wil_acc, wil_size, wil_lat], ignore_index=True)
wilcoxon_df.to_csv(cfg.results_dir / 'reduced_wilcoxon.csv', index=False)
wilcoxon_df

## 6. Plots

Latency-vs-features Pareto, method × metric comparison table. Cross-dataset stability
(`unsw_nb15` overlay) is deliberately skipped here — it adds another ~45 minutes. If you
want it, copy section 6 of `run_colab.ipynb` into a new cell below.

In [ ]:
from src.plots import plot_latency_vs_features, render_comparison_table

plot_latency_vs_features(df, cfg.results_dir)
render_comparison_table(agg, cfg.results_dir)

import pathlib
print('Plots written under:', cfg.results_dir)
for p in sorted(pathlib.Path(cfg.results_dir).glob('*.png'))[:20]:
    print(' ', p.name)

## 7. (Optional) Push results to GitHub

Per-attack-class SHAP signatures on the winning subset are deliberately deferred —
`compute_shap_signatures()` needs a fitted LCCDE and an `X_sample` DataFrame, not the
aggregate matrix. Run it in a follow-up notebook by re-fitting LCCDE on the `bgwo_shap`
winning feature subset and calling `compute_shap_signatures(model, X_sample, 100)`.

In [ ]:
import getpass
PAT = getpass.getpass('Paste GitHub PAT (leave blank to skip): ')
if PAT:
    REPO  = input('owner/repo [yazanjer/IDS-features-selection]: ').strip() or 'yazanjer/IDS-features-selection'
    EMAIL = input('git email [yazan.aljeroudi@gmail.com]: ').strip() or 'yazan.aljeroudi@gmail.com'
    NAME  = input('git name [Yazan Aljeroudi]: ').strip() or 'Yazan Aljeroudi'
    !git config user.email "$EMAIL"
    !git config user.name  "$NAME"
    !git add results/ && git commit -m 'Reduced-budget Colab run (n_seeds=5, pop=10, iter=20)' || true
    !git remote set-url origin "https://${PAT}@github.com/${REPO}.git"
    !git push origin HEAD
    # Best-effort token scrub from the remote URL (keep the working tree clean).
    !git remote set-url origin "https://github.com/${REPO}.git"
else:
    print('Skipping GitHub push.')